# YAML-to-Fortran Pipeline with advanced debug outputs

This notebook demonstrates the full YAML → Fortran pipeline with the `ACGOrchestrator` and intentionally includes multiple, detailed debug outputs. Those debug diagnostics are part of its main value: they help inspect mapping decisions, preprocessing behavior, and code-generation internals.

## Setup

Import required modules and configure notebook-relative paths.

Note: this notebook includes detailed debug output by design.

In [1]:
import sys
import warnings
from pathlib import Path
import yaml
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Notebook-relative project paths (no user-specific absolute paths)
notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    project_root = Path('..')
else:
    project_root = Path('.')

# Make project importable
sys.path.insert(0, str(project_root))

# Import ACG modules
from acg_brns.acg_orchestrator import ACGOrchestrator
from acg_brns.formula_evaluator import FormulaEvaluator
from acg_brns.yaml_to_acg_mapper import YAMLtoACGMapper

# Tracks whether the pipeline is still on track; downstream cells check this
# instead of raising, so the notebook can be run through even after a failure.
pipeline_ok = True

print(f"Project root (relative): {project_root}")
print(f"Working directory       : {notebook_dir.name}")


## Configure Paths

Define input YAML file and output directory for Fortran files.

In [2]:
# Input YAML configuration (relative to project root)
yaml_path = project_root / 'models' / 'canfield_equilibrium.yaml'

# Output directory for Fortran files (relative to project root)
output_dir = project_root / 'generated_fortran' / 'equilibrium'

# Warn instead of raising, so the rest of the notebook can still run through
if not yaml_path.exists():
    warnings.warn(f"YAML file not found: {yaml_path}. Later cells will be skipped.")
    pipeline_ok = False
else:
    print(f"✓ Input YAML: {yaml_path}")

print(f"✓ Output directory: {output_dir}")

# Create output directory
output_dir.mkdir(parents=True, exist_ok=True)
print("✓ Output directory created/verified")


## Load and Inspect YAML Configuration

Load the YAML file and display key information.

In [3]:
# Load YAML
with open(yaml_path, 'r') as f:
    config = yaml.safe_load(f)

# Display configuration summary
print("Configuration Summary:")
print("=" * 60)
print(f"Species: {len(config['species'])}")
print(f"Reactions: {len(config['reactions'])}")
print(f"Bio-parameters: {len(config['parameters']['biogeochemical'])}")
print(f"Computed values: {len(config.get('computed_values', {}))}")

# Species breakdown
dissolved = [s for s in config['species'] if s.get('type') == 'dissolved']
solid = [s for s in config['species'] if s.get('type') == 'solid']
print(f"\nSpecies breakdown:")
print(f"  Dissolved: {len(dissolved)}")
print(f"  Solid: {len(solid)}")

# Reaction types
equilibrium = [r for r in config['reactions'] if r.get('equilibrium', False)]
kinetic = [r for r in config['reactions'] if not r.get('equilibrium', False)]
print(f"\nReaction types:")
print(f"  Kinetic: {len(kinetic)}")
print(f"  Equilibrium: {len(equilibrium)}")

### Species List

In [4]:
print("Dissolved Species:")
for i, s in enumerate(dissolved, 1):
    print(f"  {i:2d}. {s['name']:10s} (D0={s.get('transport_D0', 0):.2e})")

print(f"\nSolid Species:")
for i, s in enumerate(solid, len(dissolved)+1):
    print(f"  {i:2d}. {s['name']:10s}")

### Reaction List

In [5]:
print("Reactions:")
print("=" * 80)
for rxn in config['reactions']:
    rxn_type = "[EQ]" if rxn.get('equilibrium', False) else "[KIN]"
    print(f"{rxn['id']:2d}. {rxn_type} {rxn['name']:30s}")
    
    # Show stoichiometry
    stoich = rxn.get('stoichiometry', {})
    # Convert values to float for comparison
    stoich_float = {}
    for k, v in stoich.items():
        try:
            stoich_float[k] = float(v) if isinstance(v, str) else v
        except (ValueError, TypeError):
            stoich_float[k] = 0.0
    
    reactants = [f"{v:+.1f} {k}" for k, v in stoich_float.items() if v < 0]
    products = [f"{v:+.1f} {k}" for k, v in stoich_float.items() if v > 0]
    
    if reactants and products:
        print(f"     {' + '.join([r.replace('-', '').replace('+', '') for r in reactants])} → {' + '.join([p.replace('+', '') for p in products])}")
    print()

## Create ACG Orchestrator

Initialize the orchestrator with configuration.

In [6]:
# Create orchestrator with verbose output
orchestrator = ACGOrchestrator(
    yaml_path=str(yaml_path),
    output_dir=str(output_dir),
    verbose=True
)

print("✓ Orchestrator created")

## Phase 1: Load and Validate Configuration

In [7]:
config = None
if pipeline_ok:
    try:
        config = orchestrator.load_config()
        print("\n✓ Phase 1 complete")
    except Exception as exc:
        warnings.warn(f"Phase 1 (load_config) failed: {exc}")
        pipeline_ok = False
else:
    print("⚠ Skipping Phase 1 (previous step failed).")


## Phase 2: Evaluate Formulas

Evaluate all mathematical expressions and computed values.

In [8]:
params = {}
if pipeline_ok:
    try:
        params = orchestrator.evaluate_formulas()

        # Show some evaluated parameters
        print("\nSample evaluated parameters:")
        sample_keys = list(params.keys())#[:10]
        for key in sample_keys:
            val = params[key]
            if isinstance(val, (int, float)):
                print(f"  {key:15s} = {val:.6e}")
            else:
                print(f"  {key:15s} = {val}")

        print("\n✓ Phase 2 complete")
    except Exception as exc:
        warnings.warn(f"Phase 2 (evaluate_formulas) failed: {exc}")
        pipeline_ok = False
else:
    print("⚠ Skipping Phase 2 (previous step failed).")


## Phase 3: Map to ACG Structures

Convert YAML data to ACG-compatible structures.

In [9]:
acg_data = None
if pipeline_ok:
    try:
        acg_data = orchestrator.map_to_acg_structures()

        # Display structure information
        print("\nACG Data Structures:")
        print("=" * 60)
        print(f"Variables (species): {len(acg_data['variables'])}")
        print(f"Bio-parameters: {len(acg_data['bio_name'])}")
        print(f"Reactions: {acg_data['nreactions']}")
        print(f"Stoichiometry matrix: {acg_data['stoich_matrix'].shape}")

        print("\n✓ Phase 3 complete")
    except Exception as exc:
        warnings.warn(f"Phase 3 (map_to_acg_structures) failed: {exc}")
        pipeline_ok = False
else:
    print("⚠ Skipping Phase 3 (previous step failed).")


### Visualize Stoichiometry Matrix

In [10]:
# Create stoichiometry matrix heatmap
stoich_matrix = np.array(acg_data['stoich_matrix'].tolist()).astype(float)

fig, ax = plt.subplots(figsize=(14, 10))
im = ax.imshow(stoich_matrix.T, cmap='RdBu_r', aspect='auto', vmin=-2, vmax=2)

# Set ticks and labels
ax.set_xticks(range(acg_data['nreactions']))
ax.set_xticklabels([f"R{i+1}" for i in range(acg_data['nreactions'])], rotation=45, ha='right')
ax.set_yticks(range(len(acg_data['variables'])))
ax.set_yticklabels(acg_data['variables'])

ax.set_xlabel('Reactions', fontsize=12)
ax.set_ylabel('Species', fontsize=12)
ax.set_title('Stoichiometry Matrix (Canfield Model)', fontsize=14, fontweight='bold')

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Stoichiometric Coefficient', rotation=270, labelpad=20)

plt.tight_layout()
plt.show()

# Print matrix statistics
non_zero = np.count_nonzero(stoich_matrix)
total = stoich_matrix.size
sparsity = (total - non_zero) / total * 100
print(f"\nStoichiometry Matrix Statistics:")
print(f"  Shape: {stoich_matrix.T.shape}")
print(f"  Non-zero entries: {non_zero} / {total}")
print(f"  Sparsity: {sparsity:.1f}%")

### Species Index Mapping

In [11]:
print("Species → Fortran Index Mapping:")
print("=" * 50)
for name, idx in acg_data['species_map'].items():
    species_type = "[D]" if idx <= acg_data['ndissolved'] else "[S]"
    print(f"{species_type} {name:10s} → sp({idx:2d},j)")

## Phase 4: Pre-processing (Gaussian Elimination)

Execute p0-p10 to reduce the stoichiometric system.

In [12]:
reduced_system = None
if pipeline_ok:
    try:
        reduced_system = orchestrator.run_preprocessing()

        print("\nReduced System:")
        print("=" * 60)
        print(f"Residual equations: {len(reduced_system['func'])}")
        print(f"Jacobian matrix: {reduced_system['jacobian'].shape}")
        print(f"Conservation rows: {reduced_system['conservation_rows']}")
        print(f"Pivot rows: {len(reduced_system['pivot_rows'])}")

        print("\n✓ Phase 4 complete")
    except Exception as exc:
        warnings.warn(f"Phase 4 (run_preprocessing) failed: {exc}")
        pipeline_ok = False
else:
    print("⚠ Skipping Phase 4 (previous step failed).")


### Visualize Jacobian Structure and Run Structural Checks

The checks below are intentionally **value-agnostic** where possible.
They focus on sparsity structure (e.g., empty rows/columns, zero diagonal entries) so they remain useful even when Jacobian values change between model variants.

In [13]:
# Convert Jacobian to numeric structure for visualization
jacobian = reduced_system['jacobian']
ncompo = len(acg_data['variables'])

# Check which entries are non-zero (symbolically)
jac_structure = np.zeros((ncompo, ncompo), dtype=int)
for i in range(ncompo):
    for j in range(ncompo):
        if jacobian[i, j] != 0:
            jac_structure[i, j] = 1

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(jac_structure, cmap='Blues', aspect='equal')

# Set ticks and labels
ax.set_xticks(range(ncompo))
ax.set_xticklabels(acg_data['variables'], rotation=45, ha='right')
ax.set_yticks(range(ncompo))
ax.set_yticklabels(acg_data['variables'])

ax.set_xlabel('Species (j)', fontsize=12)
ax.set_ylabel('Species (i)', fontsize=12)
ax.set_title('Jacobian Sparsity Pattern (∂f_i/∂C_j)', fontsize=14, fontweight='bold')

# Add grid
ax.set_xticks(np.arange(ncompo) - 0.5, minor=True)
ax.set_yticks(np.arange(ncompo) - 0.5, minor=True)
ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.5, alpha=0.3)

plt.tight_layout()
plt.show()

# -------------------------------
# Structural diagnostics (robust)
# -------------------------------
non_zero_jac = int(np.count_nonzero(jac_structure))
total_jac = int(jac_structure.size)
sparsity_jac = (total_jac - non_zero_jac) / total_jac * 100
diag = np.diag(jac_structure)
zero_diag_idx = [i for i, v in enumerate(diag) if v == 0]
row_nnz = jac_structure.sum(axis=1)
col_nnz = jac_structure.sum(axis=0)
empty_rows = [i for i, v in enumerate(row_nnz) if v == 0]
empty_cols = [i for i, v in enumerate(col_nnz) if v == 0]

print(f"\nJacobian Statistics:")
print(f"  Shape: {jac_structure.shape}")
print(f"  Non-zero entries: {non_zero_jac} / {total_jac}")
print(f"  Sparsity: {sparsity_jac:.1f}%")
print(f"  Zero diagonal entries: {len(zero_diag_idx)} / {ncompo}")

if zero_diag_idx:
    names = [acg_data['variables'][i] for i in zero_diag_idx]
    print(f"  Rows with J[i,i] = 0: {zero_diag_idx}")
    print(f"  Affected species: {names}")

if empty_rows:
    print(f"  ⚠ Empty Jacobian rows (no dependencies): {empty_rows}")
if empty_cols:
    print(f"  ⚠ Empty Jacobian columns (no variable influence): {empty_cols}")
if not empty_rows and not empty_cols:
    print("  ✓ No empty rows/columns in Jacobian structure")

# Optional lightweight numeric rank smoke test (value-dependent)
# This is only a heuristic and can vary with parameter values.
try:
    import random
    import sympy as sp

    free_syms = sorted(list(jacobian.free_symbols), key=lambda s: str(s))
    if free_syms:
        trials = 3
        ranks = []
        for _ in range(trials):
            subs = {s: random.uniform(0.5, 2.0) for s in free_syms}
            j_num = np.array(sp.Matrix(jacobian).subs(subs), dtype=float)
            ranks.append(int(np.linalg.matrix_rank(j_num)))
        print(f"  Rank smoke test over {trials} random substitutions: {ranks} (target: {ncompo})")
        if min(ranks) < ncompo:
            print("  ⚠ Potential rank deficiency for some parameter states")
        else:
            print("  ✓ Full rank in sampled substitutions")
except Exception as exc:
    print(f"  (Rank smoke test skipped: {exc})")

### Show Sample Residual Equations

In [14]:
from sympy import pretty

print("Sample Residual Equations (first 3):")
print("=" * 80)
for i in range(min(3, len(reduced_system['func']))):
    species = acg_data['variables'][i]
    func = reduced_system['func'][i]
    print(f"\nf_{i+1} (d{species}/dt):")
    print(f"  {func}")
    print()

## Phase 5: Generate Fortran Code

Generate all Fortran subroutines using acg0-acg17b functions.

In [15]:
result = None
if pipeline_ok:
    try:
        result = orchestrator.run_code_generation()
        print("\n✓ Phase 5 complete")
        print(f"\nGenerated {len(result['files_generated'])} Fortran files")
        print(f"\nNote: Using Maple-compatible ACG function names (acg0, acg1, ..., acg17b)")
    except Exception as exc:
        warnings.warn(f"Phase 5 (run_code_generation) failed: {exc}")
        print(f"\nNote: Some ACG functions may need additional implementation.")
        print(f"Pre-processing (Phases 1-4) completed successfully.")
        import traceback
        traceback.print_exc()
else:
    print("⚠ Skipping Phase 5 (previous step failed).")


In [16]:
# DEBUG: Check nnodes value
print(f"\nDEBUG: config['grid'] = {orchestrator.config.get('grid', {})}")
grid_cfg = orchestrator.config.get('grid', {})
nnodes_val = grid_cfg.get('nnodes', 100)
print(f"DEBUG: nnodes from YAML = {nnodes_val}")
print(f"DEBUG: nx should be = {2 * nnodes_val - 1}")

In [17]:
# Check physical_flags order
phys_flags = orchestrator.config.get('parameters', {}).get('physical_flags', {})
print(f"physical_flags keys: {list(phys_flags.keys())}")
print(f"physical_flags values: {list(phys_flags.values())}")

# Also check what orchestrator will extract
from acg_brns.acg_orchestrator import ACGOrchestrator
import inspect
source = inspect.getsource(orchestrator.run_code_generation)
# Print only the acg0 relevant lines
lines = source.split('\n')
for i, line in enumerate(lines):
    if 'phys_names2' in line or 'physical_flags' in line:
        print(f"Line {i}: {line}")

In [18]:
# Force complete module reload - clear all cached modules
import sys
for module_name in list(sys.modules.keys()):
    if 'acg_brns' in module_name:
        del sys.modules[module_name]

# Re-import fresh
import importlib
import acg_brns.acg_orchestrator
importlib.reload(acg_brns.acg_orchestrator)
from acg_brns.acg_orchestrator import ACGOrchestrator

print("✓ Modules reloaded")

## List Generated Files

Show all Fortran files created in the output directory.

In [19]:
# List all .f and .f90 files in output directory
fortran_files = sorted(output_dir.glob('*.f*')) if output_dir.exists() else []

if fortran_files:
    print(f"Generated Fortran files in {output_dir.relative_to(project_root)}:")
    print("=" * 60)
    for i, ffile in enumerate(fortran_files, 1):
        size = ffile.stat().st_size
        print(f"{i:2d}. {ffile.name:25s} ({size:7,d} bytes)")
    print(f"\nTotal: {len(fortran_files)} files")
else:
    print(f"No Fortran files found in {output_dir}")


## Summary

Display final pipeline summary.

In [20]:
print("="*80)
print("PIPELINE SUMMARY")
print("="*80)
print(f"Input YAML: {yaml_path}")
print(f"Output Directory: {output_dir}")
print()

if acg_data is not None:
    print("Model Configuration:")
    print(f"  Species: {acg_data['ncompo']} ({acg_data['ndissolved']} dissolved + {acg_data['nsolids']} solid)")
    print(f"  Reactions: {acg_data['nreactions']}")
    print(f"  Bio-parameters: {len(acg_data['bio_name'])}")
    print()
else:
    print("⚠ Model configuration unavailable (pipeline did not complete).")
    print()

if reduced_system is not None:
    print("Reduced System:")
    print(f"  Residual equations: {len(reduced_system['func'])}")
    print(f"  Jacobian matrix: {reduced_system['jacobian'].shape}")
    print(f"  Conservation laws: {len(reduced_system.get('conservation_basis', []))}")
    print()
else:
    print("⚠ Reduced system unavailable (pipeline did not complete).")
    print()

if fortran_files:
    print(f"Generated Files: {len(fortran_files)}")
    total_size = sum(f.stat().st_size for f in fortran_files)
    print(f"Total size: {total_size:,} bytes")
print("="*80)
print("✓ PIPELINE COMPLETE" if pipeline_ok else "⚠ PIPELINE INCOMPLETE (see warnings above)")
print("="*80)


## Next Steps

1. **Compile Fortran Code**: Use the BRNS Makefile to compile generated subroutines
2. **Run Simulation**: Execute BRNS with the new model
3. **Analyze Results**: Visualize concentration profiles and reaction rates